## Information Retrieval 24/25, University of Pisa
### Franco Maria Nardini, Rossano Venturini (francomaria.nardini@isti.cnr.it, rossano.venturini@unipi.it)

----

# Data Compression

### Read Documents

In [7]:
from datasets import load_dataset

c4_subset = load_dataset("allenai/c4", data_files="en/c4-train.0102*-of-01024.json.gz")

# Get a list of URls and a list of corresponding documents

urls = [x['url'] for x in c4_subset["train"]]
documents = [x['text'] for doc_id, x in enumerate(c4_subset["train"])]

print(f"Number of documents:  {len(urls)}")
print(f"Number of characters: {sum(len(x) for x in documents)}")

c4_subset = None

Number of documents:  1425269
Number of characters: 3065881920


### Read the Inverted Index created with the previous notebook

In [8]:
import pickle

filename = 'inverted_index.pkl'

with open(filename, 'rb') as file:
    N, avgdl, vocabulary_map, posting_lists, df, idf, frequencies, tf, doc_lengths = pickle.load(file)

<br><br>
---

### Dictionary compression

In [11]:
# Compute longest common prefix (lcp) between two strings.
#
# Examples:
# >>> lcp("aaa", "aaaaaa")
# 3
# >>> lcp("baa", "aaaaaa")
# 0
# >>> lcp("aba", "aaa")
# 1
def lcp(s, z):
    for i, (a, b) in enumerate(zip(s, z)):
        if a != b:
            return i
    return i+1

In [12]:
### Front-Coding
def front_coding_encode(C):
    # C is a sorted collection of strings
    fc_size = 0
    prev = None
    for s in C:
        if prev != None:
            c_lcp = lcp(s, prev)
        else:
            c_lcp = 0

        fc_size += len(s) - c_lcp
        prev = s
        yield (c_lcp, s[c_lcp:])
    return fc_size

In [13]:
def front_coding_decode(fc):
    C = []
    prev = ""
    for lcp, suff in fc:
        s = prev[:lcp] + suff
        C.append( s )
        prev = s
    return C

In [15]:
sorted_vocabulary = sorted(vocabulary_map.keys())

fc = list(front_coding_encode(sorted_vocabulary))

In [31]:
start = 2000000
end = start + 10

for orig, new in zip(sorted_vocabulary[start:end], fc[start:end]):
    print("{0:50}, {1:50}".format(orig, str(new)))

hi-cap                                            , (5, 'p')                                          
hi-capacity                                       , (6, 'acity')                                      
hi-caps                                           , (6, 's')                                          
hi-caption                                        , (6, 'tion')                                       
hi-carbon                                         , (5, 'rbon')                                       
hi-chairs                                         , (4, 'hairs')                                      
hi-chew                                           , (5, 'ew')                                         
hi-chroma                                         , (5, 'roma')                                       
hi-chrome                                         , (8, 'e')                                          
hi-claps                                          , (4, 'laps')          

In [36]:
print("No compression: {0:>10} chars".format(sum(len(s) for s in sorted_vocabulary)))
print("Fron-Coding:    {0:>10} chars".format(sum(len(s) for _,s in fc)))

No compression:   42623327 chars
Fron-Coding:      17545557 chars


In [65]:
sorted_urls = sorted(urls)

fc_urls = list(front_coding_encode(sorted_urls))

In [69]:
fc_urls[:10]

[(0,
  'http://%e2%80%8bwiping-rags-absorbents-spill-kits.americantex.com/item/colored-recycled-rags/knit-ganzie-t-shirt-material-polo-s-/pn-1625'),
 (72, 'ylinder-products/universal-cylinder-bracket/7216-ye'),
 (7, '0-www.doi.org.libus.csd.mu.edu/news/DOINewsJun09.html'),
 (8, '.everyday-families.com/parents-planning-meals/'),
 (9, 'kixify.com/adidas-adilette-cloudfoam-plus-logo-steel'),
 (8, '00ezpb.wcomhost.com/blog/?p=792'),
 (9, '1games.com/game/50201.htm'),
 (26, '1117.htm?gb'),
 (10,
  'yourtranslationservice.com/travel-Europe/country-links/Czech-Republic/witch-burning/witch-burning-explanation.html'),
 (9,
  '221.info/quartz-dining-table/best-of-quartz-dining-table-for-97-quartz-dining-table-canada/')]

In [70]:
print("No compression: {0:>10} chars".format(sum(len(s) for s in sorted_urls)))
print("Fron-Coding:    {0:>10} chars".format(sum(len(s) for _,s in fc_urls)))

No compression:   99849350 chars
Fron-Coding:      66849105 chars


<br><br>
---
## Compression of posting lists


In [1]:
## Universal codes
import math
from math import log2

def number_bits(x : int) -> int:
    if x <= 1: return 0
    return int( math.floor(math.log2( x - 1 )) + 1)

In [2]:
# Compute the length of the unary code of a given integer value x.
def unary_length(x):
    if x <= 0: raise ValueError("Encoded number must be greater than zero.")

    return x


In [3]:
# Compute the length of Variable byte code
def vbyte_length(x):
    if x <= 0: raise ValueError("Encoded number must be greater than zero.")
    x = x - 1;
    s = 0
    while True:
        s += 8
        if x < 128:            
            break
        x = x // 128

    return 8

In [4]:
# Compute the length of the Elias' Gamma code of a given integer value x.
def elias_gamma_length(x):

    if x <= 0: raise ValueError("Encoded number must be greater than zero.")
    l = int(log2(x))
    n = 1 + l
    return n+l

In [80]:
# Compute the length of the Elias' Delta code of a given integer value x.
def elias_delta_length(x):
    if x <= 0: raise ValueError("Encoded number must be greater than zero.")

    n = elias_gamma_length(1 + log2(x))
    l = int(log2(x))
    return n+l

In [81]:
# Given a sequence, the function generates the sequence of gaps.
def sequence_to_dgaps(lst):
    prev = -1
    for x in lst:
        yield x - prev
        prev = x

In [82]:
def evaluate(lists):
    no_compression = 0
    gamma = 0
    vbyte = 0
    delta = 0
    for p_list in lists:
        no_compression += number_bits(p_list[-1]+1) * len(p_list)      
        for d_gap in sequence_to_dgaps(p_list):
            vbyte += vbyte_length(d_gap)
            gamma += elias_gamma_length(d_gap)
            delta += elias_delta_length(d_gap)
    return no_compression//(8*1024**2), vbyte//(8*1024**2), gamma//(8*1024**2), delta//(8*1024**2)

In [83]:
how_many_lists = 10000

In [84]:
no_compression, vbyte, gamma, delta = evaluate(posting_lists[:how_many_lists])

print(f"No compression: {no_compression} MiB")
print(f"VByte:          {vbyte} MiB")
print(f"Elias Gamma:    {gamma} MiB")
print(f"Elias Delta:    {delta} MiB")

No compression: 337 MiB
VByte:          128 MiB
Elias Gamma:    143 MiB
Elias Delta:    137 MiB


<br>

### The impact of sorting by URLs

In [85]:
no_compression, vbyte, gamma, delta = evaluate(permuted_posting_lists[:how_many_lists])

print(f"No compression: {no_compression} MiB")
print(f"VByte:          {vbyte} MiB")
print(f"Elias Gamma:    {gamma} MiB")
print(f"Elias Delta:    {delta} MiB")

No compression: 337 MiB
VByte:          128 MiB
Elias Gamma:    133 MiB
Elias Delta:    128 MiB


## TODO:
- Plot distribution of dgaps before and after reordering

<br><br>

#### Reorder doc_ids sorting by URLs

### Let's compute the number of distinct domains
We have a small sample, we want to be sure large enough clusters.

In [87]:
def get_domain(url):
    url = url.replace("https://", "").replace("http://", "")
    domain = url.split("/")[0]
    return domain

In [88]:
domains = {}

for url in urls:
    domain = get_domain(url)
    if domain not in domains:
        domains[domain] = 0
    domains[domain] += 1

print(f"Number of domains: {len(domains)}")
print(f"Number of domains with at least 4 pages: {sum([1 for (domain, x) in domains.items() if x >= 4])}")

Number of domains: 841520
Number of domains with at least 4 pages: 49076


**Goal**: Assign close *docIds* to documents from the same domain. **Why?**

Let's reverse the domain so that:

https://microsoft.github.io --> io.github.microsoft §

In [50]:
# https://microsoft.github.io --> io.github.microsoft
def get_domain_rev(domain):
    domain = domain.replace("https://", "").replace("http://", "")
    return ".".join(domain.split(".")[::-1])

def get_url_rev(url):
    url = url.replace("https://", "").replace("http://", "")
    split = url.split("/")
    return get_domain_rev(split[0]) + "/" + "/".join(split[1:])

In [51]:
print(get_domain_rev("https://microsoft.github.io"))

io.github.microsoft


In [52]:
print(get_url_rev("https://microsoft.github.io/presidio/getting_started/"))

io.github.microsoft/presidio/getting_started/


In [53]:
print(urls[0])
print(get_url_rev(urls[0]))

https://americanhealthandbeauty.com/articles/2704/non-surgical-fat-reduction--zerona-vs-zeltiq
com.americanhealthandbeauty/articles/2704/non-surgical-fat-reduction--zerona-vs-zeltiq


In [54]:
urls_rev = [get_url_rev(url) for url in urls]

print(urls_rev[0])

com.americanhealthandbeauty/articles/2704/non-surgical-fat-reduction--zerona-vs-zeltiq


In [55]:
permutation = list(range(len(urls))) # [0, 1, 2, ....] 

permutation.sort(key = lambda i: get_url_rev(urls[i])) # sort doc_ids by URL

In [62]:
urls[permutation[0]]

'https://www.accc.gov.au./media-release/accc-holiday-operations'

In [60]:
# Let's compute the inverse permutation because we need to map an old doc_id to the new one

inv_permutation = [0]*len(permutation) 
for i, p in enumerate(permutation):
    inv_permutation[p] = i 

In [61]:
# Permute the posting lists

permuted_posting_lists = []
for p_list in posting_lists:
    permuted_posting_lists.append( sorted(inv_permutation[posting] for posting in p_list) )